In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp


print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec


# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

from analysis_village.cc1pi.CutMasks.MaskUtils import *
# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.Optimize import OptimizationUtils
from analysis_village.cc1pi.Optimize import ConfusionMatricesUtils
from analysis_village.cc1pi.BDTs import BDTTrainingUtils

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrames

In [ ]:
from cols_to_keep import *
#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data_gen1/mc_ar23p_pruned_min_reco_no_syst.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100, reprocess_df = False, reprocess_truth = False)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
#mc_bnb_evt_df = mc_bnb_evt_df[min_reco_cols_to_keep]
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']

# We only need the categories for this study and the expandable systs are really big
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
#load in time cosmic df
mc_in_time_cosmics_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data_gen1/cc1pi_5e18_in_time_cosmics.df", keys2load, 100)
mc_in_time_cosmics_evt_df = mc_in_time_cosmics_df['cc1pi']
mc_in_time_cosmics_hdr_df = mc_in_time_cosmics_df['hdr']

#Load CV lowE dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_lowE_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data_gen1/cc1pi_1e20_lowE_CV.df", keys2load, 5, filter_df = False)
mc_bnb_lowE_evt_df = mc_bnb_lowE_df['cc1pi']
mc_bnb_lowE_nu_df = mc_bnb_lowE_df['nudf']
mc_bnb_lowE_hdr_df = mc_bnb_lowE_df['hdr']
mc_bnb_lowE_nu_df = mc_bnb_lowE_nu_df

In [ ]:
from analysis_village.cc1pi.systematics.constants import *

pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
'''
data_tot_pot = data_hdr_df['pot'].sum()
data_gates = data_hdr_df.nbnbinfo.sum()
'''
print("data_tot_pot: %.3e" %(data_tot_pot))
print("data tot gates : %.3e" %(data_gates))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))

#Low E
mc_bnb_lowE_tot_pot = mc_bnb_lowE_hdr_df['pot'].sum()
print("dirt_tot_pot: %.3e" %(mc_bnb_lowE_tot_pot))
mc_bnb_lowE_pot_scale = data_tot_pot / mc_bnb_lowE_tot_pot
print("dirt_pot_scale: %.3e" %(mc_bnb_lowE_pot_scale))
mc_bnb_lowE_evt_df[pot_weight_col] = mc_bnb_lowE_pot_scale * np.ones(len(mc_bnb_lowE_evt_df))

intime_gates = mc_in_time_cosmics_hdr_df[mc_in_time_cosmics_hdr_df['first_in_subrun'] == 1]['ngenevt'].sum()
print("intime cosmics data gates: {:.2e}".format(intime_gates))
f = 0.075
scale_intime_to_lightdata = (1-f)*data_gates/intime_gates
print("goal scale: {:.2f}".format(scale_intime_to_lightdata))
mc_in_time_cosmics_evt_df[pot_weight_col] = scale_intime_to_lightdata * np.ones(len(mc_in_time_cosmics_evt_df))

In [ ]:
def is_MIP_candidate_mask(df):
    is_primary_mask = (df.pfp.parent_is_primary == True) & (df.pfp.dist_to_vertex < CTE.max_primary_distance_to_vertex)
    is_track_mask = (df.pfp.trk.len > CTE.min_track_lenght) & (df.pfp.trackScore > CTE.min_track_score)
    track_mask = is_primary_mask & is_track_mask

    len_mask = df.pfp.trk.len > CTE.MIP_candidate_min_TL
    chi2_mask = (df.pfp.trk.chi2pid.best.chi2_muon < CTE.MIP_candidate_max_muon_score) & (df.pfp.trk.chi2pid.best.chi2_proton > CTE.MIP_candidate_min_proton_score)
    return track_mask & chi2_mask & len_mask
    
def is_primary_track_mask(df):
    is_primary_mask = (df.pfp.parent_is_primary == True) & (df.pfp.dist_to_vertex < CTE.max_primary_distance_to_vertex)
    is_track_mask = (df.pfp.trk.len > CTE.min_track_lenght) & (df.pfp.trackScore > CTE.min_track_score)
    return is_primary_mask & is_track_mask

def get_muon_pion_slice_mask(df):
    """
    Returns a boolean array (aligned with df rows) that is True for slices that
    contain exactly one true muon MIP candidate and exactly one true pion MIP candidate.
    """
    SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]
    c = lambda *p: df[p + ('',) * (6 - len(p))].to_numpy()  # 6-level column keys, as in your pdg_col

    # Particle-level MIP candidate mask (same cuts as is_MIP_candidate_mask)
    is_primary = c('pfp', 'parent_is_primary').astype(bool) & (c('pfp', 'dist_to_vertex') < CTE.max_primary_distance_to_vertex)
    is_track   = (c('pfp', 'trk', 'len') > CTE.min_track_lenght) & (c('pfp', 'trackScore') > CTE.min_track_score)
    mip = (
        is_primary & is_track
        & (c('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon') < CTE.MIP_candidate_max_muon_score)
        & (c('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton') > CTE.MIP_candidate_min_proton_score)
        & (c('pfp', 'trk', 'len') > CTE.MIP_candidate_min_TL)
    )

    pdg = np.abs(c('pfp', 'trk', 'truth', 'p', 'pdg'))

    # One groupby pass to get a slice id per row, then count per slice
    ids = df.groupby(level=SLICE_LEVELS, sort=False).ngroup().to_numpy()
    n = ids.max() + 1
    muon_count = np.bincount(ids, weights=(pdg == 13) & mip, minlength=n)
    pion_count = np.bincount(ids, weights=(pdg == 211) & mip, minlength=n)

    return (muon_count[ids] == 1) & (pion_count[ids] == 1)

In [ ]:
# %% ------------------------------------------------------------------
# Imports (keep your existing ones: perform_truth_matching, CTE, etc.)
# ---------------------------------------------------------------------
import gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from analysis_village.cc1pi.Constants import CTE as CTE
 
SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]
SIDEBANDS = ("",)   # add "proton" and/or "two_pions" if you will need them later
 
K = lambda *p: p + ('',) * (6 - len(p))   # pad a column key to 6 levels
# columns kept for slices that FAIL the selection (used for the reco-"bkg" row of the confusion matrix)
REJECTED_KEYS = [K('slc', 'wgt'), K('truth', 'nu_categ'), K('truth', 'nu_categ_proton_reduced')]
 
 
# %% ------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------
def concat_shift_first_index(dfs, level=0, free_inputs=True, desc="Concatenating"):
    """
    Concatenates MultiIndex DataFrames with shifted index level, popping inputs
    sequentially to minimize peak memory overhead during concatenation.
    Takes a LIST of DataFrames.
    """
    if isinstance(level, str):
        level = dfs[0].index.names.index(level)
 
    parts = []
    offset = 0
 
    with tqdm(total=len(dfs), desc=desc, unit="df") as pbar:
        while dfs:
            df = dfs.pop(0)
            pbar.update(1)
            if df is None or df.empty:
                continue
 
            lv = df.index.levels[level]
            max_val = int(lv.max())
 
            if offset > 0:
                # Shallow copy with re-leveled index (no data duplication)
                df = df.copy(deep=False)
                df.index = df.index.set_levels(lv + offset, level=level, verify_integrity=False)
 
            parts.append(df)
            offset += max_val + 1
            pbar.set_postfix(rows=sum(len(p) for p in parts))
 
        if not parts:
            return pd.DataFrame()
 
        pbar.set_postfix_str("running pd.concat...")
        out = pd.concat(parts, copy=False)
 
    parts.clear()
    gc.collect()
    return out
 
 
def final_selection_mask(df, sidebands=("",)):
    """
    Boolean array (one entry per row): True where the slice passes ALL cuts of at least
    one of the requested selections. Equivalent to OR over sidebands of
    build_event_cumulative_masks(df, sb)[last stage], without copying the slc columns.
    """
    def get(*p):
        return df[p + ('',) * (6 - len(p))].to_numpy()
 
    def cut(name):
        return get('slc', 'cut', name).astype(bool)   # NaN would become True: make sure cut columns have no NaN
 
    head = (cut('obvious_cosmic') & cut('t0') & cut('inside_FV')
            & (get('slc', 'nu_score') > CTE.min_nu_score) & cut('track'))
    containment = (get('slc', 'cut_var', 'n_exiting_pfps') == 0) & (get('slc', 'cut', 'no_high_yz') == True)
    tail = containment & cut('TPC_containment') & cut('michel')
 
    keep = np.zeros(len(df), dtype=bool)
    for sb in sidebands:
        if sb == "two_pions":
            m = (head & (get('slc', 'cut_var', 'n_MIP_candidates') > 2) & cut('shower') & cut('angle')
                 & cut('proton_BDT_2pi') & tail & cut('energy'))
        else:
            bdt = cut('proton_BDT_sideband') if sb == "proton" else cut('proton_BDT')
            m = (head & cut('MIP_candidates') & cut('shower') & cut('angle')
                 & bdt & tail & cut('extra_pion') & cut('energy'))
        keep |= m
    return keep
 
 
def split_by_selection(df, name="df", sidebands=("",)):
    """
    Splits df (needs the truth columns, so call it AFTER truth matching) into
      kept : all rows of the slices passing the full selection (copy of only those rows)
      rej  : tiny table, ONE ROW PER REJECTED SLICE, with only weight + true categories.
             It lets the confusion matrix keep its reco-"bkg" row (slices failing the selection).
    """
    keep = final_selection_mask(df, sidebands)
    n_keep, n_rows = int(keep.sum()), len(keep)
 
    rej = df[REJECTED_KEYS].iloc[np.flatnonzero(~keep)]          # only 3 columns
    other = [n for n in rej.index.names if n not in SLICE_LEVELS]
    slice_key = rej.index.droplevel(other) if other else rej.index
    rej = rej[~slice_key.duplicated()].reset_index(drop=True)    # one row per rejected slice
 
    print(f"{name}: kept {n_keep} / {n_rows} rows ({n_keep / max(n_rows, 1):.2%}); "
          f"{len(rej)} rejected slices summarized")
    return df.iloc[np.flatnonzero(keep)], rej
 
 
def get_muon_pion_slice_mask(df):
    """
    Boolean array (aligned with df rows): True for slices that contain exactly one
    true muon MIP candidate and exactly one true pion MIP candidate.
    """
    c = lambda *p: df[p + ('',) * (6 - len(p))].to_numpy()
 
    is_primary = c('pfp', 'parent_is_primary').astype(bool) & (c('pfp', 'dist_to_vertex') < CTE.max_primary_distance_to_vertex)
    is_track = (c('pfp', 'trk', 'len') > CTE.min_track_lenght) & (c('pfp', 'trackScore') > CTE.min_track_score)
    mip = (
        is_primary & is_track
        & (c('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon') < CTE.MIP_candidate_max_muon_score)
        & (c('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton') > CTE.MIP_candidate_min_proton_score)
        & (c('pfp', 'trk', 'len') > CTE.MIP_candidate_min_TL)
    )
 
    pdg = np.abs(c('pfp', 'trk', 'truth', 'p', 'pdg'))
 
    ids = df.groupby(level=SLICE_LEVELS, sort=False).ngroup().to_numpy()
    n = ids.max() + 1
    muon_count = np.bincount(ids, weights=(pdg == 13) & mip, minlength=n)
    pion_count = np.bincount(ids, weights=(pdg == 211) & mip, minlength=n)
 
    return (muon_count[ids] == 1) & (pion_count[ids] == 1)
 
 
def reduce_to_slices(df, mask, levels=tuple(SLICE_LEVELS), chunk_slices=2000,
                     mask_col=('slc', 'cut', 'is_perfect_cc1pi', '', '', '')):
    """
    Equivalent to: df[mask_col] = mask; df.groupby(levels).first()
    but done in chunks of slices so peak memory stays near 1x the frame.
    """
    levels = list(levels)
    ids = df.groupby(level=levels).ngroup().to_numpy()   # slice ids, in sorted key order
    n = int(ids.max()) + 1
    order = np.argsort(ids, kind="stable")                # rows per slice, original order kept
    sorted_ids = ids[order]
    edges = np.searchsorted(sorted_ids, np.arange(0, n + chunk_slices, chunk_slices))
    first_row = order[np.searchsorted(sorted_ids, np.arange(n))]
    slice_mask = np.asarray(mask)[first_row]              # mask is constant within a slice
    del ids, sorted_ids
 
    parts = []
    for lo, hi in tqdm(list(zip(edges[:-1], edges[1:])), desc="slices -> chunks", unit="chunk"):
        if hi == lo:
            continue
        parts.append(df.iloc[order[lo:hi]].groupby(level=levels).first())
    out = pd.concat(parts)
    parts.clear()
    gc.collect()
    out[mask_col] = slice_mask
    return out

In [ ]:

 
 
# %% ------------------------------------------------------------------
# Pipeline
# ---------------------------------------------------------------------
# 1) BNB: truth matching, then keep only slices passing the full selection
mc_evt_df = mc_bnb_evt_df
del mc_bnb_evt_df
gc.collect()
 
if "ar23p" in bnb_path:
    print("NOT MATCHING")
else:
    mc_evt_df = perform_truth_matching(mc_evt_df, mc_bnb_nu_df)
    del mc_bnb_nu_df
gc.collect()
print("Finished loading")
 
# (truth columns exist from here on) rebinding the name frees the full frame right after the split
mc_evt_df, rejected_bnb = split_by_selection(mc_evt_df, "BNB", SIDEBANDS)
gc.collect()
 
# 2) lowE + in-time cosmics: concat, truth matching, selection split
print("Starting concat for cosmics")
dfs = [mc_bnb_lowE_evt_df, mc_in_time_cosmics_evt_df]
del mc_bnb_lowE_evt_df, mc_in_time_cosmics_evt_df   # drop names so the function can free memory
mc_bnb_lowE_evt_df = concat_shift_first_index(dfs, desc="lowE + cosmics")
del dfs
gc.collect()
 
print("TM for BNB nu df starting")
mc_bnb_lowE_evt_df = perform_truth_matching(mc_bnb_lowE_evt_df, mc_bnb_lowE_nu_df)
del mc_bnb_lowE_nu_df
gc.collect()
print("TM for BNB nu df done")
 
mc_bnb_lowE_evt_df, rejected_lowE = split_by_selection(mc_bnb_lowE_evt_df, "lowE + cosmics", SIDEBANDS)
gc.collect()
 
# 3) Perfect-CC1pi slice flag (only on the selected rows) + one row per slice (chunked to limit memory)
mask = get_muon_pion_slice_mask(mc_bnb_lowE_evt_df)
mc_bnb_lowE_evt_df = reduce_to_slices(mc_bnb_lowE_evt_df, mask)
del mask
gc.collect()
 
# 4) Final concat of BNB + (lowE + cosmics): selected slices only, one row per slice
dfs = [mc_evt_df, mc_bnb_lowE_evt_df]
del mc_evt_df, mc_bnb_lowE_evt_df
slc_df = concat_shift_first_index(dfs, desc="BNB + lowE")
del dfs
gc.collect()
 
# 5) Small table of the rejected slices (weight + true categories) for the reco-"bkg" row
rejected_df = pd.concat([rejected_bnb, rejected_lowE], ignore_index=True)
del rejected_bnb, rejected_lowE
print(f"slc_df: {len(slc_df)} selected slices | rejected_df: {len(rejected_df)} rejected slices")

In [ ]:

def plot_confusion_matrix(
    cm,
    x_labels,   # True labels
    y_labels,   # Reco labels
    cmap=sunset_cmap,
):
    cm = np.asarray(cm, dtype=float)

    # -------------------------
    # Normalizations
    # -------------------------
    # Purity (row-normalized): Sum over True for a given Reco
    row_denom = cm.sum(axis=1, keepdims=True)
    cm_purity = np.divide(cm, row_denom, where=row_denom > 0) * 100

    # Efficiency (column-normalized): Sum over Reco for a given True
    col_denom = cm.sum(axis=0, keepdims=True)
    cm_eff = np.divide(cm, col_denom, where=col_denom > 0) * 100

    # Eff * Pur
    cm_effpur = cm_purity * cm_eff / 100.0

    # -------------------------
    # Plot
    # -------------------------
    fig, axes = plt.subplots(
        1, 3, figsize=(18, 5), sharey=True, constrained_layout=True
    )

    matrices = [
        (cm_purity, "Purity", True),
        (cm_eff, "Efficiency", True),
        (cm_effpur, "Eff × Pur", False),
    ]

    for ax, (cm_plot, title, show_counts) in zip(axes, matrices):
        im = ax.imshow(cm_plot, cmap=cmap, vmin=0, vmax=100, origin="lower", aspect='auto')

        # FIX: Use specific lengths for each axis based on the actual labels provided
        ax.set_xticks(np.arange(len(x_labels)))
        ax.set_yticks(np.arange(len(y_labels)))
        
        ax.set_xticklabels(x_labels, rotation=45)
        ax.set_yticklabels(y_labels)

        ax.set_xlabel("True")
        ax.set_title(title)

        # Loop through the matrix shape, not the label list length
        for i in range(cm.shape[0]):    # Row index (Reco)
            for j in range(cm.shape[1]): # Column index (True)
                value = cm_plot[i, j]

                if show_counts:
                    txt = f"{int(cm[i,j])}\n{value:.1f}%"
                else:
                    txt = f"{value:.1f}%"

                text_color = "white" if value > 60 else "black"
                ax.text(
                    j, i, txt,
                    ha="center", va="center",
                    color=text_color,
                    fontsize=10
                )

    axes[0].set_ylabel("Reco")
    cbar = fig.colorbar(im, ax=axes.ravel().tolist())
    cbar.set_label("Percentage (%)")

    plt.show()

In [ ]:
from analysis_village.cc1pi.HelperFunctions import HelperFunctions

In [ ]:

def IsNu(df):
    is_numu = abs(df.pdg) == 14
    is_nue = abs(df.pdg) == 12
    return is_numu | is_nue   
    
def TruthInFV(data):
    xmin = -200 + CTE.min_distance_to_wall_x_y
    xmax = 200 - CTE.min_distance_to_wall_x_y
    ymin = -200 + CTE.min_distance_to_wall_x_y
    ymax = 200 - CTE.min_distance_to_wall_x_y
    zmin = CTE.min_distance_to_first_z_wall
    zmax = 500 - CTE.min_distance_to_last_z_wall
    
    pass_fv = (data.x > xmin) & (data.x < xmax) & (data.y > ymin) & (data.y < ymax) & (data.z < zmax) & (data.z > zmin)
    return pass_fv

def isCC1Pi(df): # definition
    is_1pi1mu = (df.nmu_P_100MeV_3000MeV == 1) & (df.npi_P_130MeV_800MeV == 1) & (df.npi_P_85MeV_10000MeV == 1)
    is_NpiNmuNnNp = df.nprim - df.nmu - df.npi - df.np - df.nn == 0

    # Initialize full theta mask (False by default)
    is_theta = pd.Series(False, index=df.index)

    # Only compute angles where needed
    df_sel = df.loc[is_1pi1mu]
    
    if len(df_sel) > 0:
        cpi_vec = df_sel.loc[:, ('cpi','genp',['x','y','z'])].to_numpy()
        mu_vec  = df_sel.loc[:, ('mu','genp',['x','y','z'])].to_numpy()
     

        mu_mag  = np.linalg.norm(mu_vec, axis=1)
        cpi_mag = np.linalg.norm(cpi_vec, axis=1)
        dot     = np.sum(mu_vec * cpi_vec, axis=1)

        cos_theta = dot / np.clip(mu_mag * cpi_mag, 1e-12, None)
        theta     = np.arccos(np.clip(cos_theta, -1.0, 1.0))
        
        # Assign back using the SAME index subset
        is_theta.loc[df_sel.index] = theta < CTE.max_angle_between_candidates
    
    #return is_1pi1mu & is_NpiNmuNnNp & is_theta & is_mu_contained
    return is_1pi1mu & is_NpiNmuNnNp & is_theta

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

K = lambda *p: p + ('',) * (6 - len(p))   # pad a column key to 6 levels

file_dir = "/exp/sbnd/data/users/lpelegri/Graphs/ProtonSeparation"
os.makedirs(file_dir, exist_ok=True)
dpi = 300

pot_weight_col = K('slc', 'wgt')

# slc_df must already be slice-level (one row per slice); otherwise weights would be double counted.
assert slc_df.index.is_unique, "slc_df has several rows per slice: reduce it to one row per slice first"


def select(df, keys, mask=None):
    """Small copy with only the requested columns (and only the rows where mask is True)."""
    sub = df[list(keys)]
    return sub if mask is None else sub[mask]

In [ ]:
proto_key = K('truth', 'nu_categ_proton_reduced')

weighted_distribution = (
    slc_df[pot_weight_col]
    .groupby(slc_df[proto_key])
    .sum()
    .sort_values(ascending=False)
)

print("--- Weighted Distribution of the selected sample ---")
print(f"{'Category':<25} | {'Weighted Events':>15}")
print("-" * 45)
for category, weight in weighted_distribution.items():
    print(f"{str(category):<25} | {weight:>15.2f}")

print("-" * 45)
print(f"{'TOTAL':<25} | {weighted_distribution.sum():>15.2f}")

In [ ]:
HelperFunctions.print_purity(slc_df, ('truth','nu_categ','','','',''))

In [ ]:
true_categ_key   = K('truth', 'nu_categ')
true_protons_key = K('truth', 'nu_categ_proton_reduced')
reco_protons_key = K('slc', 'measure_var', 'num_protons')

labels        = ["bkg", "0p_CC1Pi", "1p_CC1Pi", "plus2p_CC1Pi"]
pretty_labels = ["Background", r"$CC1\pi 0p$", r"$CC1\pi 1p$", r"$CC1\pi 2p+$"]

# Selected slices (slc_df) followed by rejected slices (rejected_df, they all go to reco "bkg")
is_cc1pi   = np.concatenate([(slc_df[true_categ_key] == "CC1pi").to_numpy(),
                             (rejected_df[true_categ_key] == "CC1pi").to_numpy()])
nu_protons = np.concatenate([slc_df[true_protons_key].to_numpy(),
                             rejected_df[true_protons_key].to_numpy()])
weights    = np.concatenate([slc_df[pot_weight_col].to_numpy(),
                             rejected_df[pot_weight_col].to_numpy()])
passes_cuts = np.concatenate([np.ones(len(slc_df), dtype=bool),
                              np.zeros(len(rejected_df), dtype=bool)])
n_p_reco   = np.concatenate([slc_df[reco_protons_key].to_numpy(dtype=float),
                             np.full(len(rejected_df), np.nan)])

true_cat = np.select([
    ~is_cc1pi,
    is_cc1pi & (nu_protons == "0p_CC1Pi"),
    is_cc1pi & (nu_protons == "1p_CC1Pi"),
    is_cc1pi & (nu_protons == "plus2p_CC1Pi"),
], labels, default="bkg")

reco_cat = np.select([
    ~passes_cuts,
    passes_cuts & (n_p_reco == 0),
    passes_cuts & (n_p_reco == 1),
    passes_cuts & (n_p_reco > 1),
], labels, default="bkg")

# rows = true category, columns = reco category
weighted_cm = (
    pd.Series(weights)
    .groupby([true_cat, reco_cat]).sum()
    .unstack(fill_value=0)
    .reindex(index=labels, columns=labels, fill_value=0)
    .to_numpy()
)

fig = ConfusionMatricesUtils.plot_confusion_matrix(weighted_cm.T, pretty_labels, y_labels=pretty_labels)
fig.savefig(file_dir + "/confusion_matrix_proton_selection", bbox_inches='tight', dpi=dpi)
plt.show()

# Pion Separation Stuff

In [ ]:

print(dpi)
# 6. Save and Show
file_dir = "/exp/sbnd/data/users/lpelegri/Graphs/MuonPionSeparation"
os.makedirs(file_dir, exist_ok=True)

In [ ]:
true_labels        = ["other", "muon", "pion", "proton", "shower"]
pretty_true_labels = ["other", r"$\mu$", r"$\pi$", "p", r"$e/\gamma$"]
reco_labels        = ["muon", "pion"]
pretty_reco_labels = [r"$\mu$", r"$\pi$"]
reco_label_to_idx  = {l: i for i, l in enumerate(reco_labels)}


def get_clean_ids(series, valid_labels):
    s = series.replace(['inelastic pion', 'stopping pion'], 'pion')
    return s.where(s.isin(valid_labels), "other")


reco_tasks = [
    ('muon', K('slc', 'measure_var', 'mu_true_p_type')),
    ('pion', K('slc', 'measure_var', 'pi_true_p_type')),
]
contained = slc_df[K('slc', 'measure_var', 'muon_contained')]
scenarios = [                                   # (title, row mask or None, file suffix)
    ("All",       None,                "all"),
    ("Contained", contained == True,   "contained"),
    ("Exiting",   contained == False,  "exiting"),
]

for title_suffix, mask, file_suffix in scenarios:
    print(f"\n--- Processing Scenario: {title_suffix} ---")
    sel = select(slc_df, [pot_weight_col] + [k for _, k in reco_tasks], mask)
    if sel.empty:
        print(f"Warning: No events found for scenario {title_suffix}. Skipping.")
        continue

    weighted_cm = np.zeros((len(true_labels), len(reco_labels)))
    for r_label, true_p_type_key in reco_tasks:
        counts = sel[pot_weight_col].groupby(get_clean_ids(sel[true_p_type_key], true_labels)).sum()
        weighted_cm[:, reco_label_to_idx[r_label]] += counts.reindex(true_labels, fill_value=0).to_numpy()

    fig = ConfusionMatricesUtils.plot_confusion_matrix(weighted_cm.T, pretty_true_labels, y_labels=pretty_reco_labels)
    fig.suptitle(f"Muon/Pion Separation ({title_suffix})", fontsize=16)

    save_path = os.path.join(file_dir, f"confusion_matrix_muon_pion_separation_{file_suffix}.pdf")
    fig.savefig(save_path, bbox_inches='tight', dpi=dpi)
    print(f"Saved: {save_path}")
    plt.show()

# Perfect Sample

In [ ]:
true_labels        = ["muon", "pion"]
pretty_true_labels = [r"$\mu$", r"$\pi$"]
reco_labels        = ["muon", "pion"]
pretty_reco_labels = [r"$\mu$", r"$\pi$"]
reco_label_to_idx  = {l: i for i, l in enumerate(reco_labels)}

perfect_cc1pi = (slc_df[K('slc', 'cut', 'is_perfect_cc1pi')] == True) & (slc_df[K('truth', 'nu_categ')] == "CC1pi")
contained = slc_df[K('slc', 'measure_var', 'muon_contained')]
scenarios = [
    ("All",       perfect_cc1pi,                          "all"),
    ("Contained", perfect_cc1pi & (contained == True),    "contained"),
    ("Exiting",   perfect_cc1pi & (contained == False),   "exiting"),
]

for title_suffix, mask, file_suffix in scenarios:
    print(f"\n--- Processing Scenario: {title_suffix} ---")
    sel = select(slc_df, [pot_weight_col] + [k for _, k in reco_tasks], mask)
    if sel.empty:
        print(f"Warning: No events found for scenario {title_suffix}. Skipping.")
        continue

    weighted_cm = np.zeros((len(true_labels), len(reco_labels)))
    for r_label, true_p_type_key in reco_tasks:
        counts = sel[pot_weight_col].groupby(get_clean_ids(sel[true_p_type_key], true_labels)).sum()
        weighted_cm[:, reco_label_to_idx[r_label]] += counts.reindex(true_labels, fill_value=0).to_numpy()

    fig = ConfusionMatricesUtils.plot_confusion_matrix(weighted_cm.T, pretty_true_labels, y_labels=pretty_reco_labels)
    fig.suptitle(f"Muon/Pion Separation ({title_suffix})", fontsize=16)

    save_path = os.path.join(file_dir, f"confusion_matrix_muon_pion_separation_only_mu_pi_{file_suffix}.pdf")
    fig.savefig(save_path, bbox_inches='tight', dpi=dpi)
    print(f"Saved: {save_path}")
    plt.show()

# In perfect sample evaluate the efficiency as a function of the lenght

In [ ]:
save_fig_dir = file_dir
muon_mass = 105.66
pion_mass = 139.57

keys = {
    'wgt':        pot_weight_col,
    'mu_pdg':     K('slc', 'measure_var', 'mu_true_pdg'),
    'pi_pdg':     K('slc', 'measure_var', 'pi_true_pdg'),
    'mu_p':       K('slc', 'measure_var', 'mu_true_p'),
    'pi_p':       K('slc', 'measure_var', 'pi_true_p'),
    'reco_p_mu':  K('slc', 'measure_var', 'reco_p_mu'),
    'range_p_pi': K('slc', 'measure_var', 'range_p_pi'),
}

perfect_cc1pi = (slc_df[K('slc', 'cut', 'is_perfect_cc1pi')] == True) & (slc_df[K('truth', 'nu_categ')] == "CC1pi")
scenarios = [
    ("Contained", perfect_cc1pi & (slc_df[K('slc', 'measure_var', 'muon_contained')] == True), "contained"),
]


def efficiency_per_bin(values, mask, bins):
    """Fraction of events passing mask in each bin of values (binomial errors)."""
    total,   _ = np.histogram(values,       bins=bins)
    passing, _ = np.histogram(values[mask], bins=bins)
    with np.errstate(divide='ignore', invalid='ignore'):
        eff = np.where(total > 0, passing / total, np.nan)
        err = np.where(total > 0, np.sqrt(eff * (1 - eff) / total), np.nan)
    return eff, err


def plot_step_with_error(ax, bin_centers, bins, eff, err, color, label):
    ax.step(bins, np.append(eff, eff[-1]), where='post', color=color, linewidth=2, label=label)
    ax.errorbar(bin_centers, eff, yerr=err, fmt='none', color=color, capsize=3, alpha=0.7)


for title_suffix, mask, file_suffix in scenarios:
    print(f"\n--- Processing Scenario: {title_suffix} ---")
    sel = select(slc_df, keys.values(), mask)
    if sel.empty:
        print(f"Warning: No events found for scenario {title_suffix}. Skipping.")
        continue
    a = {name: sel[key].to_numpy() for name, key in keys.items()}

    bins_mu = np.linspace(0.1, 1.0, 41)    # 40 bins from 0.1 to 1 GeV
    bins_pi = np.linspace(0.13, 1.0, 41)   # 40 bins from 0.13 to 1 GeV
    bin_centers_mu = 0.5 * (bins_mu[:-1] + bins_mu[1:])
    bin_centers_pi = 0.5 * (bins_pi[:-1] + bins_pi[1:])

    is_muon = np.abs(a['mu_pdg']) == 13
    is_pion = np.abs(a['pi_pdg']) == 211

    # Which candidate has the larger (mass-corrected) momentum
    is_mu_longest = a['reco_p_mu'] > a['range_p_pi'] * pion_mass / muon_mass
    is_pi_longest = a['range_p_pi'] * pion_mass / muon_mass > a['reco_p_mu']

    is_mu_primary = (is_mu_longest & is_muon) | (is_pi_longest & ~is_pion)
    is_pi_primary = (is_mu_longest & is_muon) | (is_pi_longest & ~is_pion)   # NOTE: same expression as is_mu_primary, check this is intended

    eff_mu,         err_mu         = efficiency_per_bin(a['mu_p'], is_muon,       bins_mu)
    eff_mu_primary, err_mu_primary = efficiency_per_bin(a['mu_p'], is_mu_primary, bins_mu)
    eff_pi,         err_pi         = efficiency_per_bin(a['pi_p'], is_pion,       bins_pi)
    eff_pi_primary, err_pi_primary = efficiency_per_bin(a['pi_p'], is_pi_primary, bins_pi)

    fig, (ax_mu, ax_pi) = plt.subplots(1, 2, figsize=(15, 6))

    plot_step_with_error(ax_mu, bin_centers_mu, bins_mu, eff_mu, err_mu, color='royalblue', label=r"BDT")
    plot_step_with_error(ax_mu, bin_centers_mu, bins_mu, eff_mu_primary, err_mu_primary, color='crimson', label=r"Longest is $\mu$")
    ax_mu.set_xlabel(r"True $\mu$ momentum [GeV]", fontsize=14)
    ax_mu.set_ylabel(r"Fraction of slices", fontsize=14)
    ax_mu.set_xlim(bins_mu[0], bins_mu[-1])
    ax_mu.set_ylim(0, 1.1)
    ax_mu.grid(axis='y', linestyle='--', alpha=0.5)
    ax_mu.legend(fontsize=11, loc='lower right')

    plot_step_with_error(ax_pi, bin_centers_pi, bins_pi, eff_pi, err_pi, color='royalblue', label=r"BDT")
    plot_step_with_error(ax_pi, bin_centers_pi, bins_pi, eff_pi_primary, err_pi_primary, color='crimson', label=r"Longest is $\mu$")
    ax_pi.set_xlabel(r"True $\pi$ momentum [GeV]", fontsize=14)
    ax_pi.set_ylabel(r"Fraction of slices", fontsize=14)
    ax_pi.set_xlim(bins_pi[0], bins_pi[-1])
    ax_pi.set_ylim(0, 1.1)
    ax_pi.grid(axis='y', linestyle='--', alpha=0.5)
    ax_pi.legend(fontsize=11, loc='lower right')

    plt.tight_layout()
    fig.savefig(os.path.join(save_fig_dir, f"combined_match_eff_{file_suffix}.pdf"), bbox_inches='tight')
    plt.show()